# 1. Import Libraries

In [1]:
import sys
from pathlib import Path

# Thêm thư mục gốc (project root) vào Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

import numpy as np

from utils.features import build_features

import optuna

# 2. Load Dataset

In [2]:
df_train, df_val = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = None,
                                pls_fit_years = [1]
                                   )

In [3]:
features = [
    'Year', 'Month', 'Day', 'Hour',
    'Combined_Temp',
    'Combined_GHI_1', 'Combined_GHI_2',
    'sin_day_k1', 'cos_day_k1',
    'sin_day_k2', 'cos_day_k2',
    'sin_week_k1', 'cos_week_k1',
    'dow',
    'lag_1_temp', 'lag_1_ghi_1', 'lag_1_ghi_2',
    'delta_1_temp', 'delta_1_ghi_1', 'delta_1_ghi_2',
    'delta_24_temp', 'delta_24_ghi_1', 'delta_24_ghi_2'
]
target = 'Load'

# 3. Split Data

In [4]:
# Year 1 → TRAIN
train_mask = df_train['Year'] == 1

# Year 2 → VALIDATION
val_mask = df_val['Year'] == 2

# Year 3 → TEST
# Testing.xlsx is assumed to contain Year 3

X_train = df_train.loc[train_mask, features]
y_train = df_train.loc[train_mask, target]

X_val = df_val.loc[val_mask, features]
y_val = df_val.loc[val_mask, target]

print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")

Dataset sizes:
Train (Year 1): 8,784
Validation (Year 2): 8,760


# 4. Train on year 1, evaluate on year 2

In [5]:
tscv = TimeSeriesSplit(n_splits=4)

In [6]:
def objective(trial):

    params = {
        "objective": "reg:squarederror",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10,
            log=True
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    fold_rmse = []

    # ============================================
    # Time-series CV trên YEAR 1
    # ============================================

    for train_idx, val_idx in tscv.split(X_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        model.fit(
            X_fold_train,
            y_fold_train
        )

        y_pred = model.predict(
            X_fold_val
        )

        rmse = root_mean_squared_error(
            y_fold_val,
            y_pred
        )

        fold_rmse.append(rmse)

    # Optuna minimize mean CV RMSE
    return np.mean(fold_rmse)

In [7]:
sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction="minimize",
    study_name="xgb_year1_timeseries_cv",
    sampler=sampler
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)

[I 2026-10-02 19:05:53,582] A new study created in memory with name: xgb_year1_timeseries_cv


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-10-02 19:05:58,479] Trial 0 finished with value: 265.99280166625977 and parameters: {'n_estimators': 624, 'learning_rate': 0.17254716573280354, 'max_depth': 8, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.2537815508265665}. Best is trial 0 with value: 265.99280166625977.
[I 2026-10-02 19:06:05,194] Trial 1 finished with value: 260.1857109069824 and parameters: {'n_estimators': 1092, 'learning_rate': 0.010636066512540286, 'max_depth': 10, 'min_child_weight': 9, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.12561043700013558}. Best is trial 1 with value: 260.1857109069824.
[I 2026-10-02 19:06:08,824] Trial 2 finished with value: 267.50529861450195 and parameters: {'n_estimators': 705, 'learning_rate': 0.023927528765580644, 'max_depth': 7, 'min_chil

In [8]:
print("Best CV RMSE:")
print(study.best_value)

print("\nBest parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best CV RMSE:
245.67664337158203

Best parameters:
n_estimators: 630
learning_rate: 0.14127945276792403
max_depth: 3
min_child_weight: 3
subsample: 0.6860796351063262
colsample_bytree: 0.7348490781363393
gamma: 1.0523946503484392
reg_alpha: 4.8778142331171976e-06
reg_lambda: 1.8940998726084215


In [9]:
best_params = study.best_params

seeds = [0, 1, 2, 3, 42]

results = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(X_val)

    results.append({
        "Seed": seed,
        "RMSE": root_mean_squared_error(y_val, y_pred),
        "MSE": mean_squared_error(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "MAPE": mean_absolute_percentage_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred)
    })

stability_df = pd.DataFrame(results)

stability_summary = stability_df.drop(
    columns="Seed"
).agg(["mean", "std", "min", "max"]).T

print("=== Results by seed ===")
display(stability_df)

print("\n=== Stability summary ===")
display(stability_summary)

=== Results by seed ===


,Seed,RMSE,MSE,MAE,MAPE,R2
0,0,155.215622,24091.888672,114.367210,0.053766,0.854173
1,1,153.471359,23553.457031,112.982391,0.053243,0.857432
2,2,153.910370,23688.402344,113.351936,0.053365,0.856615
3,3,155.346558,24132.550781,114.260727,0.053940,0.853927
4,42,152.568451,23277.130859,112.387375,0.053109,0.859105



=== Stability summary ===


,mean,std,min,max
RMSE,154.102472,1.180601,152.568451,155.346558
MSE,23748.685937,363.748871,23277.130859,24132.550781
MAE,113.469928,0.844691,112.387375,114.367210
MAPE,0.053484,0.000354,0.053109,0.053940
R2,0.856250,0.002202,0.853927,0.859105


In [10]:
stability_df.to_csv("../report/seed_delta24_temp_ghi_val_report.csv", index = False)
stability_summary.to_csv("../report/seed_summary_delta24_temp_ghi_val_report.csv", index = False)

# 4. Final Training on year 1 and year 2

In [11]:
df_train, df_test = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = pd.read_excel("../Dataset/testing.xlsx")
                                   )

In [12]:
X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]

In [13]:
print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")
print(f"Test (Year 3): {len(X_test):,}")

Dataset sizes:
Train (Year 1): 17,544
Validation (Year 2): 8,760
Test (Year 3): 8,760


In [14]:
seeds = [0, 1, 2, 3, 42]

test_predictions = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)

    test_predictions.append(pred)

y_test_pred = np.mean(
    test_predictions,
    axis=0
)

In [15]:
pd.DataFrame(y_test_pred).to_csv("../report/delta24_temp_ghi_test_prediction.csv", index = False)